# F1 - Preparação da Representação Intermediária (IR)

Este notebook constrói e valida a Representação Intermediária (*Intermediate Representation*, IR) utilizada nas estratégias que decompõem a tradução entre linguagem natural e Nile. A F1 recebe exclusivamente os artefatos congelados da F0 e não utiliza modelo professor, modelo aluno ou edição manual das estruturas.

A cadeia de construção é inteiramente determinística:

```text
Nile canônica da F0
        ↓
parser Lark
        ↓
Árvore de Sintaxe Abstrata (AST)
        ↓
mapeamento AST → IR
        ↓
validação por JSON Schema
        ↓
persistência da IR
        ↓
reconstrução IR → AST → Nile
```

A AST é uma estrutura transitória produzida pelo parser. A IR é o artefato persistido, serializável e fornecido às fases posteriores. Ela funciona como ponte sintática controlada: torna explícitos escopo, origem, destino, targets, operações, itens e restrições temporais sem introduzir conteúdo que não esteja na referência Nile.

Os campos formais são mantidos em inglês, incluindo `source`, `destination`, `targets`, `operations`, `operator`, `kind`, `value` e `temporal_constraint`. Os textos de apresentação, auditoria e diagnóstico permanecem em português.

A F1 não induz uma nova linguagem por modelo generativo. O modelo abstrato é definido a partir da estrutura formal disponibilizada pela F0 e do conjunto de construções efetivamente observado no CAMPI. A validade é verificada por esquema e por equivalência de ida e volta.

| Bloco | Etapa | Função metodológica | Resultado principal |
|---:|---|---|---|
| 1 | Configuração inicial | Preparar ambiente, diretórios e funções comuns | Base da F1 |
| 2 | Auditoria da F0 | Confirmar integridade e compatibilidade da entrada | Artefatos da F0 carregados |
| 3 | Modelo abstrato | Definir campos, vocabulário e regras de mapeamento | Especificação operacional da IR |
| 4 | JSON Schema | Tornar a estrutura validável automaticamente | `ir_schema.json` |
| 5 | Transformações | Implementar AST ↔ IR | `ir_core.py` |
| 6 | Geração | Produzir as 50 IRs de referência | `ir_references.jsonl` |
| 7 | Validação | Conferir registros persistidos e casos negativos | Validação integral |
| 8 | Equivalência | Verificar AST → IR → AST → Nile → AST | `ir_validation.csv` |
| 9 | Empacotamento | Registrar hashes, ambiente e inventário | `f1_operacional.zip` |

Ao final, a F1 fornece uma representação estrutural consistente com as 50 referências canônicas e pronta para ser utilizada nas justificativas da F2 e nas estratégias intermediárias da F3.

## Bloco 1 - Configuração inicial

### Objetivo

Este bloco prepara o ambiente da F1 e fixa os parâmetros utilizados na construção das IRs. A organização segue o mesmo padrão da F0 para preservar consistência visual, rastreabilidade e separação entre arquivos temporários e operacionais.

### Procedimentos executados

O bloco:

- importa bibliotecas de sistema, serialização, hashing e manipulação tabular;
- verifica a disponibilidade de `jsonschema` e realiza instalação controlada quando necessário;
- desativa bytecodes transitórios;
- configura a exibição do Pandas;
- identifica o ambiente de execução;
- define diretórios de entrada, trabalho, saída operacional e pacote final;
- registra os caminhos de `ir_schema.json`, `ir_core.py`, `ir_references.jsonl`, `ir_validation.csv` e `manifest.json`;
- fixa o conjunto CAMPI, os 50 exemplos e a versão operacional da IR;
- cria funções comuns de leitura, escrita, JSONL, hashing, importação controlada e empacotamento;
- define o padrão visual e a numeração automática das tabelas.

### Princípios da fase

A F1 não altera os artefatos da F0. Todas as transformações são derivadas das ASTs produzidas pelo parser congelado. Os arquivos de entrada são tratados como somente leitura e seus hashes são verificados antes do uso.

### Resultado esperado

Ao final, os diretórios e as funções da F1 estarão disponíveis. Nenhuma IR é produzida neste bloco e nenhuma tabela é exibida, pois a entrada ainda não foi carregada.

In [1]:
# ----------------------------------------------------------
# 1.1 Importação das bibliotecas principais
# ----------------------------------------------------------

import copy
import hashlib
import importlib
import importlib.metadata as importlib_metadata
import importlib.util
import json
import os
import platform
import py_compile
import shutil
import subprocess
import sys
import textwrap
import zipfile

from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import HTML, display


# ----------------------------------------------------------
# 1.2 Instalação controlada da dependência jsonschema
# ----------------------------------------------------------

def garantir_pacote(modulo: str, pacote_pip: str | None = None) -> None:
    """Instala um pacote somente quando o módulo ainda não está disponível."""
    pacote_pip = pacote_pip or modulo

    if importlib.util.find_spec(modulo) is not None:
        return

    resultado = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", pacote_pip],
        capture_output=True,
        text=True,
        check=False,
    )

    importlib.invalidate_caches()

    if resultado.returncode != 0:
        raise RuntimeError(
            f"Não foi possível instalar o pacote '{pacote_pip}'. "
            f"Erro retornado pelo pip: {resultado.stderr.strip()}"
        )


garantir_pacote("jsonschema", "jsonschema")


# ----------------------------------------------------------
# 1.3 Importações dependentes da verificação anterior
# ----------------------------------------------------------

import jsonschema
from jsonschema import Draft202012Validator


# ----------------------------------------------------------
# 1.4 Configuração de exibição e escrita de bytecode
# ----------------------------------------------------------

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 180)
pd.set_option("display.width", 0)

sys.dont_write_bytecode = True


# ----------------------------------------------------------
# 1.5 Definição do ambiente e dos diretórios principais
# ----------------------------------------------------------

KAGGLE_INPUT_DIR = Path("/kaggle/input")
KAGGLE_WORKING_DIR = Path("/kaggle/working")

BASE_DIR = Path(
    os.environ.get(
        "F1_OUTPUT_ROOT",
        str(KAGGLE_WORKING_DIR if KAGGLE_WORKING_DIR.exists() else Path.cwd())
    )
).resolve()

F1_DIR = BASE_DIR / "f1_operacional"
ZIP_F1_PATH = BASE_DIR / "f1_operacional.zip"
F0_EXTRACT_DIR = BASE_DIR / "_f0_operacional_extraido"

if F1_DIR.exists():
    shutil.rmtree(F1_DIR)

F1_DIR.mkdir(parents=True, exist_ok=True)

if ZIP_F1_PATH.exists():
    ZIP_F1_PATH.unlink()


# ----------------------------------------------------------
# 1.6 Definição dos caminhos dos artefatos da F1
# ----------------------------------------------------------

IR_SCHEMA_PATH = F1_DIR / "ir_schema.json"
IR_CORE_PATH = F1_DIR / "ir_core.py"
IR_REFERENCES_PATH = F1_DIR / "ir_references.jsonl"
IR_VALIDATION_PATH = F1_DIR / "ir_validation.csv"
MANIFEST_PATH = F1_DIR / "manifest.json"


# ----------------------------------------------------------
# 1.7 Parâmetros fixos da fase
# ----------------------------------------------------------

FASE = "F1"
DATASET_ID = "CAMPI"
EXPECTED_EXAMPLES = 50
IR_VERSION = "1.0.0"
EXPECTED_OUTPUT_FILES = 5


# ----------------------------------------------------------
# 1.8 Funções auxiliares de leitura, escrita e hash
# ----------------------------------------------------------

def salvar_json(objeto, caminho: Path) -> None:
    caminho = Path(caminho)
    caminho.parent.mkdir(parents=True, exist_ok=True)
    with caminho.open("w", encoding="utf-8") as arquivo:
        json.dump(objeto, arquivo, ensure_ascii=False, indent=2)


def ler_json(caminho: Path):
    with Path(caminho).open("r", encoding="utf-8") as arquivo:
        return json.load(arquivo)


def salvar_jsonl(registros, caminho: Path) -> None:
    caminho = Path(caminho)
    caminho.parent.mkdir(parents=True, exist_ok=True)
    with caminho.open("w", encoding="utf-8") as arquivo:
        for registro in registros:
            arquivo.write(json.dumps(registro, ensure_ascii=False) + "\n")


def ler_jsonl(caminho: Path):
    registros = []
    with Path(caminho).open("r", encoding="utf-8") as arquivo:
        for numero_linha, linha in enumerate(arquivo, start=1):
            linha = linha.strip()
            if not linha:
                continue
            try:
                registros.append(json.loads(linha))
            except json.JSONDecodeError as exc:
                raise ValueError(
                    f"JSON inválido em {caminho}, linha {numero_linha}: {exc}"
                ) from exc
    return registros


def salvar_texto(texto: str, caminho: Path) -> None:
    caminho = Path(caminho)
    caminho.parent.mkdir(parents=True, exist_ok=True)
    caminho.write_text(str(texto), encoding="utf-8")


def calcular_sha256(caminho: Path) -> str:
    digest = hashlib.sha256()
    with Path(caminho).open("rb") as arquivo:
        for bloco in iter(lambda: arquivo.read(1024 * 1024), b""):
            digest.update(bloco)
    return digest.hexdigest()


def calcular_sha256_texto(texto: str) -> str:
    return hashlib.sha256(texto.encode("utf-8")).hexdigest()


def versao_pacote(nome: str) -> str:
    try:
        return importlib_metadata.version(nome)
    except importlib_metadata.PackageNotFoundError:
        return "nao_instalado"


def carregar_modulo_python(caminho: Path, nome_modulo: str):
    caminho = Path(caminho)
    if not caminho.exists():
        raise FileNotFoundError(f"Módulo Python não encontrado: {caminho}")

    especificacao = importlib.util.spec_from_file_location(nome_modulo, caminho)
    if especificacao is None or especificacao.loader is None:
        raise ImportError(f"Não foi possível criar a especificação para: {caminho}")

    modulo = importlib.util.module_from_spec(especificacao)
    sys.modules[nome_modulo] = modulo
    especificacao.loader.exec_module(modulo)
    return modulo


# ----------------------------------------------------------
# 1.9 Exibição padronizada e numeração das tabelas
# ----------------------------------------------------------

CONTADOR_TABELAS = 0

ROTULOS_COLUNAS = {
    "fase": "fase",
    "dataset": "conjunto de dados",
    "origem_f0": "origem da F0",
    "manifesto_f0": "manifesto da F0",
    "manifesto_sha256": "SHA-256 do manifesto",
    "arquivo": "arquivo",
    "obrigatorio": "obrigatório",
    "existe": "existe",
    "hash_registrado": "hash registrado",
    "hash_observado": "hash observado",
    "hash_confere": "hash confere",
    "tamanho_bytes": "tamanho em bytes",
    "status": "status",
    "registros": "registros",
    "ids_unicos": "IDs únicos",
    "referencias_validas": "referências válidas",
    "referencias_ida_volta": "referências com ida e volta válida",
    "campo_ir": "campo da IR",
    "tipo": "tipo",
    "obrigatoriedade": "obrigatoriedade",
    "valores_controlados": "valores controlados",
    "origem_ast": "origem na AST",
    "descricao": "descrição",
    "categoria": "categoria",
    "valor": "valor",
    "ocorrencias": "ocorrências",
    "scope_type": "scope type",
    "operator": "operator",
    "kind": "kind",
    "constraint": "constraint",
    "unit": "unit",
    "temporal_constraint": "temporal constraint",
    "definicoes": "definições",
    "required": "campos obrigatórios",
    "additional_properties": "propriedades adicionais",
    "schema_version": "versão do esquema",
    "funcao": "função",
    "entrada": "entrada",
    "saida": "saída",
    "finalidade": "finalidade",
    "id": "ID",
    "intent_id": "intent ID",
    "source": "source",
    "destination": "destination",
    "targets": "targets",
    "n_targets": "quantidade de targets",
    "n_operations": "quantidade de operations",
    "operators": "operators",
    "n_items": "quantidade de itens",
    "has_temporal": "possui temporal constraint",
    "ir_sha256": "SHA-256 da IR",
    "schema_valid": "esquema válido",
    "schema_error_count": "erros de esquema",
    "schema_feedback": "diagnóstico do esquema",
    "ast_equivalent": "AST equivalente",
    "nile_roundtrip_ok": "ida e volta Nile válida",
    "canonical_nile_equal": "Nile canônica preservada",
    "casos": "casos",
    "aprovados": "aprovados",
    "resultado_esperado": "resultado esperado",
    "resultado_observado": "resultado observado",
    "exemplos": "exemplos",
    "irs_validas": "IRs válidas",
    "equivalencias_ast_ir": "equivalências AST-IR",
    "idas_e_voltas_nile": "idas e voltas Nile",
    "arquivos_no_zip": "arquivos no ZIP",
    "zip": "ZIP",
}

ROTULOS_VALORES = {
    "deterministic_ast_to_ir": "transformação determinística AST para IR",
    "deterministic_ir_to_ast": "transformação determinística IR para AST",
    "schema_validation": "validação por JSON Schema",
    "for": "for",
    "route": "route",
    "route_for": "route + for",
    "positive": "positivo",
    "negative": "negativo",
}


def traduzir_valores_para_exibicao(df: pd.DataFrame) -> pd.DataFrame:
    """Prepara uma cópia do DataFrame apenas para apresentação visual."""
    df_exibicao = df.copy()

    def converter_valor(valor):
        if isinstance(valor, (bool, np.bool_)):
            return "sim" if valor else "não"

        if valor is None:
            return "-"

        try:
            if not isinstance(valor, str) and pd.isna(valor):
                return "-"
        except Exception:
            pass

        if isinstance(valor, str):
            valor_limpo = valor.strip()
            valor_lower = valor_limpo.lower()

            if valor_lower == "true":
                return "sim"
            if valor_lower == "false":
                return "não"
            if valor_lower in {
                "nan", "none", "null", "nat",
                "não se aplica", "nao se aplica", "n/a",
            }:
                return "-"

            return ROTULOS_VALORES.get(valor_limpo, valor)

        return valor

    for coluna in df_exibicao.columns:
        df_exibicao[coluna] = df_exibicao[coluna].apply(converter_valor)

    df_exibicao = df_exibicao.rename(
        columns={
            coluna: ROTULOS_COLUNAS.get(coluna, str(coluna).replace("_", " "))
            for coluna in df_exibicao.columns
        }
    )

    return df_exibicao


def exibir_tabela(
    df: pd.DataFrame,
    titulo: str = None,
    altura_px: int = None,
    largura_px: int = None,
    mostrar_indice: bool = False,
) -> None:
    """Exibe DataFrames com título numerado, cabeçalho fixo e rolagem."""
    global CONTADOR_TABELAS

    if df is None:
        print("Tabela não disponível.")
        return

    if not isinstance(df, pd.DataFrame):
        df = pd.DataFrame(df)

    if df.empty:
        print("Tabela vazia.")
        return

    CONTADOR_TABELAS += 1
    tabela = traduzir_valores_para_exibicao(df)

    titulo_final = f"Tabela {CONTADOR_TABELAS}"
    if titulo:
        titulo_final += f". {titulo}"

    html_tabela = tabela.to_html(
        escape=True,
        index=mostrar_indice,
        border=0,
        justify="left",
        classes="tabela_saida",
    )

    largura_css = f"{largura_px}px" if largura_px is not None else "100%"
    altura_css = (
        "overflow-y: visible;"
        if altura_px is None
        else f"max-height: {altura_px}px; overflow-y: auto;"
    )

    display(HTML(f"""
    <div style="
        font-weight: 600;
        font-size: 15px;
        margin-top: 8px;
        margin-bottom: 6px;
        color: #f1f1f1;
    ">
        {titulo_final}
    </div>

    <div class="container_tabela_saida" style="
        display: inline-block;
        width: {largura_css};
        max-width: 100%;
        overflow-x: auto;
        {altura_css}
        box-sizing: border-box;
        padding: 0;
        margin-top: 8px;
        margin-bottom: 12px;
        border: none;
        border-radius: 0;
    ">
        <style>
            .container_tabela_saida {{
                box-sizing: border-box !important;
            }}

            .container_tabela_saida table.tabela_saida {{
                border-collapse: collapse !important;
                table-layout: auto !important;
                width: auto !important;
                min-width: unset !important;
                max-width: none !important;
                font-family: Arial, sans-serif !important;
                font-size: 13px !important;
                background-color: #111 !important;
                color: #f1f1f1 !important;
                border: 1px solid #555 !important;
            }}

            .container_tabela_saida table.tabela_saida thead th {{
                position: sticky !important;
                top: 0 !important;
                z-index: 2 !important;
                background-color: #2b2b2b !important;
                color: #ffffff !important;
                font-weight: bold !important;
                padding: 7px !important;
                text-align: left !important;
                white-space: nowrap !important;
                border: 1px solid #777 !important;
                border-bottom: 2px solid #888 !important;
            }}

            .container_tabela_saida table.tabela_saida tbody td {{
                padding: 7px !important;
                vertical-align: top !important;
                text-align: left !important;
                white-space: nowrap !important;
                border: 1px solid #555 !important;
            }}

            .container_tabela_saida table.tabela_saida tbody tr:nth-child(even) td {{
                background-color: #1b1b1b !important;
            }}

            .container_tabela_saida table.tabela_saida tbody tr:nth-child(odd) td {{
                background-color: #111 !important;
            }}
        </style>
        {html_tabela}
    </div>
    """))


# ----------------------------------------------------------
# 1.10 Saída do bloco
# ----------------------------------------------------------

print("Bloco 1 concluído")
print(f"Fase: {FASE}")
print(f"Conjunto de dados esperado: {DATASET_ID}")
print(f"Versão da IR: {IR_VERSION}")
print(f"Diretório de saída: {F1_DIR}")
print("Status: OK")

Bloco 1 concluído
Fase: F1
Conjunto de dados esperado: CAMPI
Versão da IR: 1.0.0
Diretório de saída: /kaggle/working/f1_operacional
Status: OK


## Bloco 2 - Carga e auditoria dos artefatos da F0

### Objetivo

Este bloco localiza a base operacional da F0 e confirma que os arquivos utilizados para construir a IR são íntegros e compatíveis com o CAMPI.

### Localização

O diretório operacional é procurado nos inputs montados pelo Kaggle e nos caminhos informados pela configuração. A identificação exige a presença do manifesto e dos artefatos obrigatórios, evitando o uso de uma pasta incompleta.

### Conferência do manifesto

O bloco lê `manifest.json` e verifica:

- identificação da fase F0;
- identificação do conjunto CAMPI;
- quantidade esperada de exemplos;
- inventário dos arquivos;
- tamanho e SHA-256 dos artefatos registrados.

A conferência é realizada sobre os arquivos efetivamente localizados. Divergências interrompem a execução.

### Artefatos carregados

São consumidos:

- `campi_canonical.csv`;
- `nile_subset.lark`;
- `nile_core.py`;
- `validation_references.csv`;
- `manifest.json`.

O módulo Nile é importado de maneira controlada a partir da cópia congelada.

### Auditoria das referências

As 50 expressões canônicas são novamente submetidas ao parser e ao verificador estrutural. O bloco confirma:

- presença dos 50 IDs;
- unicidade dos registros;
- aceitação sintática;
- validade estrutural;
- coerência entre arquivo canônico e resultados registrados pela F0.

### Resultado esperado

As tabelas apresentam a origem dos artefatos e a situação das 50 referências. A F1 só avança quando todos os hashes e todas as referências são confirmados.

In [2]:
# ----------------------------------------------------------
# 2.1 Localização do diretório operacional da F0
# ----------------------------------------------------------

def manifesto_eh_f0(caminho: Path) -> bool:
    try:
        manifesto = ler_json(caminho)
    except Exception:
        return False

    return (
        manifesto.get("fase") == "F0"
        and manifesto.get("dataset", {}).get("id") == DATASET_ID
    )


def localizar_raiz_f0() -> Path:
    caminho_manual = os.environ.get("F0_ARTIFACTS_DIR")
    if caminho_manual:
        raiz = Path(caminho_manual).expanduser().resolve()
        manifesto = raiz / "manifest.json"
        if not raiz.is_dir() or not manifesto_eh_f0(manifesto):
            raise FileNotFoundError(
                "F0_ARTIFACTS_DIR não aponta para um diretório operacional válido da F0: "
                f"{raiz}"
            )
        return raiz

    raizes_busca = []
    if KAGGLE_INPUT_DIR.exists():
        raizes_busca.append(KAGGLE_INPUT_DIR)
    raizes_busca.append(Path.cwd())

    candidatos = []
    for raiz_busca in raizes_busca:
        for manifesto in raiz_busca.rglob("manifest.json"):
            if manifesto_eh_f0(manifesto):
                candidatos.append(manifesto.parent.resolve())

    candidatos = sorted(set(candidatos), key=lambda p: (len(str(p)), str(p)))

    if len(candidatos) == 1:
        return candidatos[0]

    if len(candidatos) > 1:
        preferidos = [
            caminho for caminho in candidatos
            if "f0-operacional" in str(caminho).lower()
            or "f0_operacional" in str(caminho).lower()
        ]
        if len(preferidos) == 1:
            return preferidos[0]

        raise RuntimeError(
            "Foram encontrados vários diretórios válidos da F0. "
            "Defina F0_ARTIFACTS_DIR com o caminho correto:\n"
            + "\n".join(str(c) for c in candidatos)
        )

    candidatos_zip = []
    for raiz_busca in raizes_busca:
        candidatos_zip.extend(raiz_busca.rglob("f0_operacional*.zip"))

    candidatos_zip = sorted({c.resolve() for c in candidatos_zip if c.is_file()})

    if len(candidatos_zip) != 1:
        raise FileNotFoundError(
            "Os artefatos da F0 não foram encontrados. Adicione o Dataset "
            "f0_operacional ao notebook do Kaggle ou defina F0_ARTIFACTS_DIR."
        )

    if F0_EXTRACT_DIR.exists():
        shutil.rmtree(F0_EXTRACT_DIR)
    F0_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(candidatos_zip[0], "r") as arquivo_zip:
        arquivo_zip.extractall(F0_EXTRACT_DIR)

    manifestos_extraidos = [
        caminho for caminho in F0_EXTRACT_DIR.rglob("manifest.json")
        if manifesto_eh_f0(caminho)
    ]

    if len(manifestos_extraidos) != 1:
        raise RuntimeError(
            "O ZIP da F0 não contém exatamente um manifesto operacional válido."
        )

    return manifestos_extraidos[0].parent.resolve()


F0_DIR = localizar_raiz_f0()
F0_MANIFEST_PATH = F0_DIR / "manifest.json"
f0_manifest = ler_json(F0_MANIFEST_PATH)


# ----------------------------------------------------------
# 2.2 Conferência integral dos hashes registrados na F0
# ----------------------------------------------------------

registros_integridade = []

for item in f0_manifest.get("files", []):
    caminho = F0_DIR / item["arquivo"]
    existe = caminho.is_file()
    hash_observado = calcular_sha256(caminho) if existe else None
    hash_confere = existe and hash_observado == item.get("sha256")

    registros_integridade.append({
        "arquivo": item["arquivo"],
        "obrigatorio": True,
        "existe": existe,
        "hash_registrado": item.get("sha256"),
        "hash_observado": hash_observado,
        "hash_confere": hash_confere,
        "tamanho_bytes": caminho.stat().st_size if existe else None,
    })

if not registros_integridade:
    raise ValueError("O manifesto da F0 não contém a lista de arquivos produzidos.")

df_integridade_f0 = pd.DataFrame(registros_integridade)

if not df_integridade_f0["hash_confere"].all():
    divergencias = df_integridade_f0.loc[~df_integridade_f0["hash_confere"]]
    raise RuntimeError(
        "A auditoria de integridade da F0 encontrou divergências:\n"
        + divergencias.to_string(index=False)
    )


# ----------------------------------------------------------
# 2.3 Verificação dos artefatos obrigatórios para a F1
# ----------------------------------------------------------

F0_CAMPI_PATH = F0_DIR / "campi_canonical.csv"
F0_GRAMMAR_PATH = F0_DIR / "nile_subset.lark"
F0_NILE_CORE_PATH = F0_DIR / "nile_core.py"
F0_REFERENCE_VALIDATION_PATH = F0_DIR / "validation_references.csv"

artefatos_obrigatorios = [
    F0_CAMPI_PATH,
    F0_GRAMMAR_PATH,
    F0_NILE_CORE_PATH,
    F0_REFERENCE_VALIDATION_PATH,
    F0_MANIFEST_PATH,
]

faltantes = [str(caminho) for caminho in artefatos_obrigatorios if not caminho.is_file()]
if faltantes:
    raise FileNotFoundError(
        "A F0 não contém todos os artefatos exigidos pela F1:\n"
        + "\n".join(faltantes)
    )


# ----------------------------------------------------------
# 2.4 Carga do corpus, da gramática e do módulo de validação Nile
# ----------------------------------------------------------

df_campi = pd.read_csv(F0_CAMPI_PATH)
df_validacao_referencias = pd.read_csv(F0_REFERENCE_VALIDATION_PATH)
gramatica_nile = F0_GRAMMAR_PATH.read_text(encoding="utf-8")

nile_core = carregar_modulo_python(F0_NILE_CORE_PATH, "f0_nile_core_f1")
validator_nile = nile_core.NileValidator(gramatica_nile)


# ----------------------------------------------------------
# 2.5 Auditoria das 50 referências operacionais
# ----------------------------------------------------------

colunas_campi_obrigatorias = {
    "id", "nl", "nile_canonical", "primary_family",
    "has_temporal", "has_route", "n_operations", "n_items",
}

colunas_ausentes = sorted(colunas_campi_obrigatorias - set(df_campi.columns))
if colunas_ausentes:
    raise KeyError(
        "Colunas obrigatórias ausentes em campi_canonical.csv: "
        + ", ".join(colunas_ausentes)
    )

if len(df_campi) != EXPECTED_EXAMPLES:
    raise ValueError(
        f"Quantidade de registros do CAMPI: {len(df_campi)}. "
        f"Esperado: {EXPECTED_EXAMPLES}."
    )

if df_campi["id"].nunique() != EXPECTED_EXAMPLES:
    raise ValueError("Os identificadores do CAMPI não são únicos.")

if set(df_campi["id"]) != set(df_validacao_referencias["id"]):
    raise ValueError("Os IDs do corpus e da validação de referências não coincidem.")

if not df_validacao_referencias["valid"].astype(bool).all():
    raise ValueError("A F0 contém referência operacional não válida.")

if not df_validacao_referencias["roundtrip_ok"].astype(bool).all():
    raise ValueError("A F0 contém referência sem ida e volta válida.")


# ----------------------------------------------------------
# 2.6 Tabelas de auditoria da entrada
# ----------------------------------------------------------

resumo_f0 = pd.DataFrame([{
    "fase": "F0",
    "dataset": f0_manifest["dataset"]["id"],
    "origem_f0": str(F0_DIR),
    "manifesto_f0": str(F0_MANIFEST_PATH),
    "manifesto_sha256": calcular_sha256(F0_MANIFEST_PATH),
    "registros": len(df_campi),
    "ids_unicos": df_campi["id"].nunique(),
    "referencias_validas": int(df_validacao_referencias["valid"].sum()),
    "referencias_ida_volta": int(df_validacao_referencias["roundtrip_ok"].sum()),
    "status": "OK",
}])

exibir_tabela(
    resumo_f0,
    "Resumo dos artefatos operacionais carregados da F0",
    altura_px=230,
)

exibir_tabela(
    df_integridade_f0,
    "Integridade dos arquivos registrados no manifesto da F0",
    altura_px=520,
)


# ----------------------------------------------------------
# 2.7 Saída do bloco
# ----------------------------------------------------------

print("Bloco 2 concluído")
print(f"Diretório da F0: {F0_DIR}")
print(f"Arquivos conferidos por hash: {len(df_integridade_f0)}")
print(f"Referências válidas carregadas: {len(df_campi)}")
print("Status: OK")

fase,conjunto de dados,origem da F0,manifesto da F0,SHA-256 do manifesto,registros,IDs únicos,referências válidas,referências com ida e volta válida,status
F0,CAMPI,/kaggle/input/datasets/thiagoarajoguedes/f0-operacional,/kaggle/input/datasets/thiagoarajoguedes/f0-operacional/manifest.json,53f33d7efd81d2f63243c467e6f67c151f52f4c330935cb13488ba05f8ba4f3d,50,50,50,50,OK


arquivo,obrigatório,existe,hash registrado,hash observado,hash confere,tamanho em bytes
biblioteca_prompts.json,sim,sim,14276ebd3ff20aad7ca29e0f61a2a7c27b5a4eb374a17b528705cc79559b9a9d,14276ebd3ff20aad7ca29e0f61a2a7c27b5a4eb374a17b528705cc79559b9a9d,sim,6445
campi_canonical.csv,sim,sim,6cd2f7fa0fcd5a8b35529cb0a3249217d53d51a642d96b743231f711f1cb3b1d,6cd2f7fa0fcd5a8b35529cb0a3249217d53d51a642d96b743231f711f1cb3b1d,sim,21184
campi_changelog.csv,sim,sim,d820fd24f09c928b6a24f8b250226a40eb17ba86128d5e9dceeebabbaa3a09e7,d820fd24f09c928b6a24f8b250226a40eb17ba86128d5e9dceeebabbaa3a09e7,sim,2848
extraction_campi.json,sim,sim,590161a042b50f4bb18177bae83a7a3da353025715a93780746d8905e9427ede,590161a042b50f4bb18177bae83a7a3da353025715a93780746d8905e9427ede,sim,70755
folds.csv,sim,sim,e4ac1579ced000e9f4272e0085ed1dc20c2328a4c00e0b4322eb4b5c743e5a75,e4ac1579ced000e9f4272e0085ed1dc20c2328a4c00e0b4322eb4b5c743e5a75,sim,11200
nile_core.py,sim,sim,5b474f77a1de738df105eccfea71454c272da6ea1922ed4dfd400e0f392613f7,5b474f77a1de738df105eccfea71454c272da6ea1922ed4dfd400e0f392613f7,sim,20106
nile_metrics.py,sim,sim,fdd9a197ac0d79023e3940df28e308bb68def931f4a489795cb35b2707f37f1b,fdd9a197ac0d79023e3940df28e308bb68def931f4a489795cb35b2707f37f1b,sim,10022
nile_subset.lark,sim,sim,427b02e12e25f79a15f8a34f1f97c9d7e383bd82ec63a1ab244a35b4d3311a5e,427b02e12e25f79a15f8a34f1f97c9d7e383bd82ec63a1ab244a35b4d3311a5e,sim,1332
prompts/f2_geracao_r1.txt,sim,sim,aa6502549e55be222eefbc996670be8e76761973a9868ee4429a9b6072266b35,aa6502549e55be222eefbc996670be8e76761973a9868ee4429a9b6072266b35,sim,3275
prompts/f2_geracao_r2.txt,sim,sim,10c6d733f92c095d07679e0cca54ed969d6b2ece040b9223afcca19d02cdb696,10c6d733f92c095d07679e0cca54ed969d6b2ece040b9223afcca19d02cdb696,sim,3261


Bloco 2 concluído
Diretório da F0: /kaggle/input/datasets/thiagoarajoguedes/f0-operacional
Arquivos conferidos por hash: 21
Referências válidas carregadas: 50
Status: OK


## Bloco 3 - Definição do modelo abstrato da IR

### Objetivo

Este bloco define quais informações da AST serão persistidas na IR e estabelece o vocabulário controlado utilizado pelos 50 exemplos.

### Análise das ASTs

As ASTs das referências são examinadas para identificar:

- formas de escopo;
- campos de origem e destino;
- tipos de target;
- operadores;
- categorias de itens;
- argumentos de operações;
- valores e unidades;
- presença de restrição temporal.

A análise é descritiva e determinística. Ela não depende de modelo generativo.

### Estrutura abstrata

A IR representa, quando aplicável:

- `intent_id`;
- `scope`;
- `source`;
- `destination`;
- `targets`;
- `operations`;
- `operator`;
- `items`;
- `kind`;
- argumentos e valores;
- `temporal_constraint`.

Campos não aplicáveis são tratados conforme as regras do modelo, sem preencher conteúdo artificial.

### Vocabulário controlado

O bloco registra os valores formais aceitos para escopos, operadores, kinds, constraints e unidades. Esse vocabulário é derivado do subconjunto operacional e limita o espaço de estruturas válidas.

### Regras de mapeamento

São documentadas regras como:

```text
AST de escopo → objeto scope
AST de target → elemento de targets
AST de operação → objeto em operations
restrição temporal → temporal_constraint
```

As regras devem preservar a informação necessária à reconstrução da AST.

### Saídas de auditoria

As tabelas apresentam:

- campos do modelo;
- valores do vocabulário;
- cobertura observada no CAMPI;
- regras de transformação.

### Resultado esperado

O bloco produz a especificação conceitual usada pelo JSON Schema e pelo módulo de transformação, sem ainda persistir as 50 IRs.

In [3]:
# ----------------------------------------------------------
# 3.1 Análise das ASTs produzidas para as referências do CAMPI
# ----------------------------------------------------------

asts_por_id = {}
contagem_scope = Counter()
contagem_operator = Counter()
contagem_kind = Counter()
contagem_constraint = Counter()
contagem_unit = Counter()
contagem_temporal = Counter()

for linha in df_campi.itertuples(index=False):
    resultado = validator_nile.validate(linha.nile_canonical)

    if not resultado["valid"]:
        raise RuntimeError(
            f"A referência {linha.id} não foi aceita pelo validador da F0: "
            f"{resultado['feedback']}"
        )

    ast_referencia = resultado["ast"]
    asts_por_id[linha.id] = ast_referencia

    scope = ast_referencia["scope"]
    contagem_scope[scope["type"]] += 1

    for operation in ast_referencia["operations"]:
        contagem_operator[operation["operator"]] += 1
        for item in operation["items"]:
            contagem_kind[item["kind"]] += 1
            if "constraint" in item:
                contagem_constraint[item["constraint"]] += 1
            if "unit" in item:
                contagem_unit[item["unit"]] += 1

    contagem_temporal[
        "present" if ast_referencia.get("interval") is not None else "absent"
    ] += 1


# ----------------------------------------------------------
# 3.2 Vocabulário controlado da IR
# ----------------------------------------------------------

IR_VOCABULARY = {
    "scope_types": ["for", "route", "route_for"],
    "target_kinds": sorted(nile_core.NileValidator.ALLOWED_TARGET_KINDS),
    "route_endpoint_kind": "endpoint",
    "operators": ["add", "allow", "block", "set", "unset"],
    "middlebox_kinds": ["middlebox"],
    "match_kinds": sorted(nile_core.NileValidator.ALLOWED_MATCH_KINDS),
    "policy_kinds": ["quota", "bandwidth"],
    "quota_constraints": sorted(nile_core.NileValidator.QUOTA_CONSTRAINTS),
    "bandwidth_constraints": sorted(nile_core.NileValidator.BANDWIDTH_CONSTRAINTS),
    "quota_units": sorted(nile_core.NileValidator.QUOTA_UNITS),
    "bandwidth_units": sorted(nile_core.NileValidator.BANDWIDTH_UNITS),
}


# ----------------------------------------------------------
# 3.3 Registro das regras de mapeamento AST para IR
# ----------------------------------------------------------

modelo_ir = pd.DataFrame([
    {
        "campo_ir": "intent_id",
        "tipo": "string",
        "obrigatoriedade": "obrigatório",
        "valores_controlados": "identificador Nile válido",
        "origem_ast": "$.intent_id",
        "descricao": "Identificador formal da intenção.",
    },
    {
        "campo_ir": "scope.type",
        "tipo": "string",
        "obrigatoriedade": "obrigatório",
        "valores_controlados": "for | route | route_for",
        "origem_ast": "$.scope.type",
        "descricao": "Forma estrutural do escopo.",
    },
    {
        "campo_ir": "scope.source",
        "tipo": "object | null",
        "obrigatoriedade": "condicional",
        "valores_controlados": "endpoint ou null",
        "origem_ast": "$.scope.from",
        "descricao": "Endpoint source quando existe rota.",
    },
    {
        "campo_ir": "scope.destination",
        "tipo": "object | null",
        "obrigatoriedade": "condicional",
        "valores_controlados": "endpoint ou null",
        "origem_ast": "$.scope.to",
        "descricao": "Endpoint destination quando existe rota.",
    },
    {
        "campo_ir": "scope.targets",
        "tipo": "array",
        "obrigatoriedade": "obrigatório",
        "valores_controlados": "endpoint | group | traffic",
        "origem_ast": "$.scope.targets",
        "descricao": "Targets associados ao escopo for.",
    },
    {
        "campo_ir": "operations[].operator",
        "tipo": "string",
        "obrigatoriedade": "obrigatório",
        "valores_controlados": "add | allow | block | set | unset",
        "origem_ast": "$.operations[].operator",
        "descricao": "Operador formal da operação.",
    },
    {
        "campo_ir": "operations[].items",
        "tipo": "array",
        "obrigatoriedade": "obrigatório",
        "valores_controlados": "dependente do operator",
        "origem_ast": "$.operations[].items",
        "descricao": "Argumentos estruturados da operação.",
    },
    {
        "campo_ir": "temporal_constraint",
        "tipo": "object | null",
        "obrigatoriedade": "obrigatório",
        "valores_controlados": "start/end em HH:MM ou null",
        "origem_ast": "$.interval",
        "descricao": "Restrição temporal simplificada.",
    },
])


# ----------------------------------------------------------
# 3.4 Cobertura observada no CAMPI
# ----------------------------------------------------------

linhas_vocabulario = []

for categoria, contador in [
    ("scope type", contagem_scope),
    ("operator", contagem_operator),
    ("kind", contagem_kind),
    ("constraint", contagem_constraint),
    ("unit", contagem_unit),
    ("temporal constraint", contagem_temporal),
]:
    for valor, ocorrencias in sorted(contador.items()):
        linhas_vocabulario.append({
            "categoria": categoria,
            "valor": valor,
            "ocorrencias": ocorrencias,
        })

df_vocabulario_observado = pd.DataFrame(linhas_vocabulario)


# ----------------------------------------------------------
# 3.5 Tabelas do modelo e do vocabulário
# ----------------------------------------------------------

exibir_tabela(
    modelo_ir,
    "Modelo abstrato da Representação Intermediária",
    altura_px=470,
)

exibir_tabela(
    df_vocabulario_observado,
    "Vocabulário estrutural observado nas 50 referências do CAMPI",
    altura_px=520,
)


# ----------------------------------------------------------
# 3.6 Saída do bloco
# ----------------------------------------------------------

print("Bloco 3 concluído")
print(f"ASTs analisadas: {len(asts_por_id)}")
print(f"Tipos de scope observados: {sorted(contagem_scope)}")
print(f"Operators observados: {sorted(contagem_operator)}")
print("Status: OK")

campo da IR,tipo,obrigatoriedade,valores controlados,origem na AST,descrição
intent_id,string,obrigatório,identificador Nile válido,$.intent_id,Identificador formal da intenção.
scope.type,string,obrigatório,for | route | route_for,$.scope.type,Forma estrutural do escopo.
scope.source,object | null,condicional,endpoint ou null,$.scope.from,Endpoint source quando existe rota.
scope.destination,object | null,condicional,endpoint ou null,$.scope.to,Endpoint destination quando existe rota.
scope.targets,array,obrigatório,endpoint | group | traffic,$.scope.targets,Targets associados ao escopo for.
operations[].operator,string,obrigatório,add | allow | block | set | unset,$.operations[].operator,Operador formal da operação.
operations[].items,array,obrigatório,dependente do operator,$.operations[].items,Argumentos estruturados da operação.
temporal_constraint,object | null,obrigatório,start/end em HH:MM ou null,$.interval,Restrição temporal simplificada.


categoria,valor,ocorrências
scope type,for,48
scope type,route,1
scope type,route + for,1
operator,add,26
operator,allow,15
operator,block,11
operator,set,15
operator,unset,2
kind,bandwidth,7
kind,middlebox,29


Bloco 3 concluído
ASTs analisadas: 50
Tipos de scope observados: ['for', 'route', 'route_for']
Operators observados: ['add', 'allow', 'block', 'set', 'unset']
Status: OK


## Bloco 4 - Construção do JSON Schema da IR

### Objetivo

Este bloco transforma o modelo abstrato em uma especificação executável. O JSON Schema define quais objetos são aceitos como IRs válidas e permite validar automaticamente as saídas de referência e, posteriormente, as IRs geradas pelo modelo aluno.

### Elementos especificados

O esquema define:

- campos obrigatórios e opcionais;
- tipos de dados;
- enums do vocabulário controlado;
- estrutura de escopos;
- targets;
- operações e itens;
- argumentos associados a cada `kind`;
- valores e unidades;
- formato das restrições temporais;
- proibição de propriedades adicionais não previstas.

Regras condicionais impedem combinações incompatíveis, como campos de rota em escopos que não utilizam origem e destino ou argumentos inadequados para determinado item.

### Verificação do próprio esquema

Antes de ser utilizado, o schema é validado como um JSON Schema bem formado. Essa etapa evita que um erro na especificação produza falsos resultados nas validações posteriores.

### Artefato produzido

O arquivo é salvo como:

```text
ir_schema.json
```

A serialização é estável, permitindo cálculo de hash e comparação nas fases seguintes.

### Resumo executável

O bloco gera uma tabela com as principais propriedades, domínios controlados e cardinalidades. Esse resumo serve como auditoria, mas o arquivo JSON é a especificação operacional oficial.

### Resultado esperado

O schema deve ser válido e conter todas as construções observadas na F0. Qualquer inconsistência impede a criação das IRs.

In [4]:
# ----------------------------------------------------------
# 4.1 Definição do JSON Schema da IR
# ----------------------------------------------------------

IR_SCHEMA = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "$id": "https://nl-nile.local/schemas/ir-v1.schema.json",
    "title": "Representação Intermediária controlada para o subconjunto Nile do CAMPI",
    "description": (
        "Esquema da IR derivada deterministicamente da AST das referências "
        "Nile do CAMPI."
    ),
    "type": "object",
    "additionalProperties": False,
    "required": [
        "intent_id",
        "scope",
        "operations",
        "temporal_constraint",
    ],
    "properties": {
        "intent_id": {
            "type": "string",
            "minLength": 1,
            "pattern": "^[A-Za-z_][A-Za-z0-9_]*$",
        },
        "scope": {"$ref": "#/$defs/scope"},
        "operations": {
            "type": "array",
            "minItems": 1,
            "items": {"$ref": "#/$defs/operation"},
        },
        "temporal_constraint": {
            "oneOf": [
                {"type": "null"},
                {"$ref": "#/$defs/temporal_constraint"},
            ]
        },
    },
    "$defs": {
        "nonempty_string": {
            "type": "string",
            "minLength": 1,
            "pattern": "^(?=.*\\S)[^'\\r\\n]+$",
        },
        "numeric_string": {
            "type": "string",
            "pattern": "^(?:0|[1-9]\\d*)(?:\\.\\d+)?$",
        },
        "hour_string": {
            "type": "string",
            "pattern": "^(?:[01]\\d|2[0-3]):[0-5]\\d$",
        },
        "endpoint_ref": {
            "type": "object",
            "additionalProperties": False,
            "required": ["kind", "value"],
            "properties": {
                "kind": {"const": "endpoint"},
                "value": {"$ref": "#/$defs/nonempty_string"},
            },
        },
        "target_ref": {
            "type": "object",
            "additionalProperties": False,
            "required": ["kind", "value"],
            "properties": {
                "kind": {"enum": IR_VOCABULARY["target_kinds"]},
                "value": {"$ref": "#/$defs/nonempty_string"},
            },
        },
        "middlebox_ref": {
            "type": "object",
            "additionalProperties": False,
            "required": ["kind", "value"],
            "properties": {
                "kind": {"const": "middlebox"},
                "value": {"$ref": "#/$defs/nonempty_string"},
            },
        },
        "match_ref": {
            "type": "object",
            "additionalProperties": False,
            "required": ["kind", "value"],
            "properties": {
                "kind": {"enum": IR_VOCABULARY["match_kinds"]},
                "value": {"$ref": "#/$defs/nonempty_string"},
            },
        },
        "quota_policy": {
            "type": "object",
            "additionalProperties": False,
            "required": ["kind", "constraint", "value", "unit"],
            "properties": {
                "kind": {"const": "quota"},
                "constraint": {"enum": IR_VOCABULARY["quota_constraints"]},
                "value": {"$ref": "#/$defs/numeric_string"},
                "unit": {"enum": IR_VOCABULARY["quota_units"]},
            },
        },
        "bandwidth_policy": {
            "type": "object",
            "additionalProperties": False,
            "required": ["kind", "constraint", "value", "unit"],
            "properties": {
                "kind": {"const": "bandwidth"},
                "constraint": {"enum": IR_VOCABULARY["bandwidth_constraints"]},
                "value": {"$ref": "#/$defs/numeric_string"},
                "unit": {"enum": IR_VOCABULARY["bandwidth_units"]},
            },
        },
        "unset_bandwidth": {
            "type": "object",
            "additionalProperties": False,
            "required": ["kind"],
            "properties": {
                "kind": {"const": "bandwidth"},
            },
        },
        "scope": {
            "oneOf": [
                {
                    "title": "for",
                    "type": "object",
                    "additionalProperties": False,
                    "required": ["type", "source", "destination", "targets"],
                    "properties": {
                        "type": {"const": "for"},
                        "source": {"type": "null"},
                        "destination": {"type": "null"},
                        "targets": {
                            "type": "array",
                            "minItems": 1,
                            "items": {"$ref": "#/$defs/target_ref"},
                        },
                    },
                },
                {
                    "title": "route",
                    "type": "object",
                    "additionalProperties": False,
                    "required": ["type", "source", "destination", "targets"],
                    "properties": {
                        "type": {"const": "route"},
                        "source": {"$ref": "#/$defs/endpoint_ref"},
                        "destination": {"$ref": "#/$defs/endpoint_ref"},
                        "targets": {
                            "type": "array",
                            "maxItems": 0,
                        },
                    },
                },
                {
                    "title": "route_for",
                    "type": "object",
                    "additionalProperties": False,
                    "required": ["type", "source", "destination", "targets"],
                    "properties": {
                        "type": {"const": "route_for"},
                        "source": {"$ref": "#/$defs/endpoint_ref"},
                        "destination": {"$ref": "#/$defs/endpoint_ref"},
                        "targets": {
                            "type": "array",
                            "minItems": 1,
                            "items": {"$ref": "#/$defs/target_ref"},
                        },
                    },
                },
            ]
        },
        "operation": {
            "oneOf": [
                {
                    "title": "add",
                    "type": "object",
                    "additionalProperties": False,
                    "required": ["operator", "items"],
                    "properties": {
                        "operator": {"const": "add"},
                        "items": {
                            "type": "array",
                            "minItems": 1,
                            "items": {"$ref": "#/$defs/middlebox_ref"},
                        },
                    },
                },
                {
                    "title": "allow",
                    "type": "object",
                    "additionalProperties": False,
                    "required": ["operator", "items"],
                    "properties": {
                        "operator": {"const": "allow"},
                        "items": {
                            "type": "array",
                            "minItems": 1,
                            "items": {"$ref": "#/$defs/match_ref"},
                        },
                    },
                },
                {
                    "title": "block",
                    "type": "object",
                    "additionalProperties": False,
                    "required": ["operator", "items"],
                    "properties": {
                        "operator": {"const": "block"},
                        "items": {
                            "type": "array",
                            "minItems": 1,
                            "items": {"$ref": "#/$defs/match_ref"},
                        },
                    },
                },
                {
                    "title": "set",
                    "type": "object",
                    "additionalProperties": False,
                    "required": ["operator", "items"],
                    "properties": {
                        "operator": {"const": "set"},
                        "items": {
                            "type": "array",
                            "minItems": 1,
                            "items": {
                                "oneOf": [
                                    {"$ref": "#/$defs/quota_policy"},
                                    {"$ref": "#/$defs/bandwidth_policy"},
                                ]
                            },
                        },
                    },
                },
                {
                    "title": "unset",
                    "type": "object",
                    "additionalProperties": False,
                    "required": ["operator", "items"],
                    "properties": {
                        "operator": {"const": "unset"},
                        "items": {
                            "type": "array",
                            "minItems": 1,
                            "items": {"$ref": "#/$defs/unset_bandwidth"},
                        },
                    },
                },
            ]
        },
        "temporal_constraint": {
            "type": "object",
            "additionalProperties": False,
            "required": ["start", "end"],
            "properties": {
                "start": {"$ref": "#/$defs/hour_string"},
                "end": {"$ref": "#/$defs/hour_string"},
            },
        },
    },
}


# ----------------------------------------------------------
# 4.2 Validação estrutural do próprio esquema
# ----------------------------------------------------------

Draft202012Validator.check_schema(IR_SCHEMA)
validador_esquema_ir = Draft202012Validator(IR_SCHEMA)


# ----------------------------------------------------------
# 4.3 Salvamento do esquema
# ----------------------------------------------------------

salvar_json(IR_SCHEMA, IR_SCHEMA_PATH)

if ler_json(IR_SCHEMA_PATH) != IR_SCHEMA:
    raise RuntimeError("O conteúdo salvo de ir_schema.json difere do esquema em memória.")


# ----------------------------------------------------------
# 4.4 Resumo executável do esquema
# ----------------------------------------------------------

resumo_esquema = pd.DataFrame([{
    "schema_version": "Draft 2020-12",
    "required": ", ".join(IR_SCHEMA["required"]),
    "additional_properties": IR_SCHEMA["additionalProperties"],
    "definicoes": len(IR_SCHEMA["$defs"]),
    "arquivo": IR_SCHEMA_PATH.name,
    "sha256": calcular_sha256(IR_SCHEMA_PATH),
    "status": "OK",
}])

exibir_tabela(
    resumo_esquema,
    "Resumo do JSON Schema da Representação Intermediária",
    altura_px=240,
)


# ----------------------------------------------------------
# 4.5 Saída do bloco
# ----------------------------------------------------------

print("Bloco 4 concluído")
print(f"Esquema salvo em: {IR_SCHEMA_PATH}")
print(f"Definições internas: {len(IR_SCHEMA['$defs'])}")
print("Status: OK")

versão do esquema,campos obrigatórios,propriedades adicionais,definições,arquivo,sha256,status
Draft 2020-12,"intent_id, scope, operations, temporal_constraint",não,13,ir_schema.json,9d34bc405a490626b09833f041dc7229ddcd40cfa12608dca6f4c58627a48d98,OK


Bloco 4 concluído
Esquema salvo em: /kaggle/working/f1_operacional/ir_schema.json
Definições internas: 13
Status: OK


## Bloco 5 - Implementação das transformações determinísticas

### Objetivo

Este bloco implementa `ir_core.py`, módulo responsável por converter AST em IR e reconstruir AST e Nile a partir da IR.

### Funções centrais

O módulo inclui funções para:

- `ast_to_ir`, que mapeia a AST para o objeto IR;
- `ir_to_ast`, que reconstrói a AST;
- validação da IR contra o schema;
- serialização canônica;
- normalização exclusivamente estrutural dos objetos;
- comparação de equivalência;
- renderização da estrutura reconstruída por meio do núcleo da F0.

### Propriedade exigida

A transformação deve preservar toda a informação formal necessária. Para cada referência, espera-se:

```text
AST original = ir_to_ast(ast_to_ir(AST original))
```

Além disso, a Nile renderizada a partir da AST reconstruída deve coincidir com a Nile canônica.

### Teste rápido

Antes de aplicar o módulo às 50 referências, o bloco seleciona um exemplo e verifica:

- criação da IR;
- conformidade com o schema;
- reconstrução da AST;
- renderização da Nile;
- igualdade com a estrutura original.

### Controle de dependências

`ir_core.py` utiliza os artefatos da F0 sem alterá-los. O arquivo é compilado e importado de forma controlada para confirmar que não contém erro sintático.

### Resultado esperado

A tabela final apresenta as funções disponíveis e sua finalidade. O módulo só é aceito quando o teste de transformação e reconstrução passa integralmente.

In [5]:
# ----------------------------------------------------------
# 5.1 Conteúdo do módulo ir_core.py
# ----------------------------------------------------------

IR_CORE_SOURCE = textwrap.dedent(r'''
from __future__ import annotations

import copy
import json
from pathlib import Path
from typing import Any, Dict, Iterable

from jsonschema import Draft202012Validator


IR_VERSION = "1.0.0"


def load_schema(path: str | Path) -> Dict[str, Any]:
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"Esquema da IR não encontrado: {path}")

    with path.open("r", encoding="utf-8") as file:
        schema = json.load(file)

    Draft202012Validator.check_schema(schema)
    return schema


def _format_path(parts: Iterable[Any]) -> str:
    path = "$"
    for part in parts:
        if isinstance(part, int):
            path += f"[{part}]"
        else:
            path += f".{part}"
    return path


def validate_ir(ir: Any, schema: Dict[str, Any]) -> Dict[str, Any]:
    validator = Draft202012Validator(schema)
    errors = []

    for error in sorted(
        validator.iter_errors(ir),
        key=lambda item: (tuple(str(part) for part in item.absolute_path), item.message),
    ):
        errors.append({
            "path": _format_path(error.absolute_path),
            "message": error.message,
            "validator": error.validator,
        })

    return {
        "schema_valid": len(errors) == 0,
        "schema_errors": errors,
    }


def assert_valid_ir(ir: Any, schema: Dict[str, Any]) -> None:
    result = validate_ir(ir, schema)
    if result["schema_valid"]:
        return

    details = "\n".join(
        f"- {error['path']}: {error['message']}"
        for error in result["schema_errors"]
    )
    raise ValueError("A IR não é válida segundo o esquema:\n" + details)


def ast_to_ir(ast: Dict[str, Any]) -> Dict[str, Any]:
    if not isinstance(ast, dict):
        raise TypeError("A AST deve ser um dicionário.")

    scope = ast.get("scope")
    operations = ast.get("operations")

    if not isinstance(scope, dict):
        raise ValueError("A AST não contém um scope válido.")
    if not isinstance(operations, list):
        raise ValueError("A AST não contém uma lista de operations.")

    interval = ast.get("interval")

    ir = {
        "intent_id": ast.get("intent_id"),
        "scope": {
            "type": scope.get("type"),
            "source": copy.deepcopy(scope.get("from")),
            "destination": copy.deepcopy(scope.get("to")),
            "targets": copy.deepcopy(scope.get("targets") or []),
        },
        "operations": copy.deepcopy(operations),
        "temporal_constraint": None,
    }

    if interval is not None:
        ir["temporal_constraint"] = {
            "start": interval["start"]["value"],
            "end": interval["end"]["value"],
        }

    return ir


def ir_to_ast(ir: Dict[str, Any]) -> Dict[str, Any]:
    if not isinstance(ir, dict):
        raise TypeError("A IR deve ser um dicionário.")

    scope = ir.get("scope")
    operations = ir.get("operations")

    if not isinstance(scope, dict):
        raise ValueError("A IR não contém um scope válido.")
    if not isinstance(operations, list):
        raise ValueError("A IR não contém uma lista de operations.")

    temporal = ir.get("temporal_constraint")

    ast = {
        "intent_id": ir.get("intent_id"),
        "scope": {
            "type": scope.get("type"),
            "from": copy.deepcopy(scope.get("source")),
            "to": copy.deepcopy(scope.get("destination")),
            "targets": copy.deepcopy(scope.get("targets") or []),
        },
        "operations": copy.deepcopy(operations),
        "interval": None,
    }

    if temporal is not None:
        ast["interval"] = {
            "start": {"kind": "hour", "value": temporal["start"]},
            "end": {"kind": "hour", "value": temporal["end"]},
        }

    return ast


def canonical_ir_json(ir: Dict[str, Any]) -> str:
    return json.dumps(
        ir,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    )


def summarize_ir(ir: Dict[str, Any]) -> Dict[str, Any]:
    scope = ir["scope"]
    source = scope.get("source")
    destination = scope.get("destination")
    operations = ir["operations"]

    return {
        "intent_id": ir["intent_id"],
        "scope_type": scope["type"],
        "source": None if source is None else source.get("value"),
        "destination": None if destination is None else destination.get("value"),
        "targets": ", ".join(
            f"{item['kind']}('{item['value']}')"
            for item in scope.get("targets", [])
        ),
        "n_targets": len(scope.get("targets", [])),
        "n_operations": len(operations),
        "operators": " | ".join(op["operator"] for op in operations),
        "n_items": sum(len(op.get("items", [])) for op in operations),
        "has_temporal": ir.get("temporal_constraint") is not None,
    }
''').strip() + "\n"

salvar_texto(IR_CORE_SOURCE, IR_CORE_PATH)


# ----------------------------------------------------------
# 5.2 Verificação sintática do módulo gerado
# ----------------------------------------------------------

ARQUIVO_PYC_TEMPORARIO = F1_DIR / "_ir_core_compile_check.pyc"

try:
    py_compile.compile(
        str(IR_CORE_PATH),
        cfile=str(ARQUIVO_PYC_TEMPORARIO),
        doraise=True,
    )
finally:
    if ARQUIVO_PYC_TEMPORARIO.exists():
        ARQUIVO_PYC_TEMPORARIO.unlink()


# ----------------------------------------------------------
# 5.3 Importação controlada do módulo
# ----------------------------------------------------------

ir_core = carregar_modulo_python(IR_CORE_PATH, "f1_ir_core_runtime")
ir_schema_runtime = ir_core.load_schema(IR_SCHEMA_PATH)


# ----------------------------------------------------------
# 5.4 Teste rápido de transformação e reconstrução
# ----------------------------------------------------------

id_teste_rapido = df_campi.iloc[0]["id"]
ast_teste_rapido = asts_por_id[id_teste_rapido]
ir_teste_rapido = ir_core.ast_to_ir(ast_teste_rapido)
ir_core.assert_valid_ir(ir_teste_rapido, ir_schema_runtime)
ast_reconstruida_teste = ir_core.ir_to_ast(ir_teste_rapido)

if ast_reconstruida_teste != ast_teste_rapido:
    raise RuntimeError("O teste rápido AST → IR → AST não preservou a estrutura.")


# ----------------------------------------------------------
# 5.5 Tabela das funções do módulo
# ----------------------------------------------------------

funcoes_modulo = pd.DataFrame([
    {
        "funcao": "ast_to_ir",
        "entrada": "AST",
        "saida": "IR",
        "finalidade": "Transformação determinística da estrutura do parser.",
    },
    {
        "funcao": "ir_to_ast",
        "entrada": "IR",
        "saida": "AST",
        "finalidade": "Reconstrução determinística da estrutura sintática abstrata.",
    },
    {
        "funcao": "validate_ir",
        "entrada": "IR + JSON Schema",
        "saida": "resultado estruturado",
        "finalidade": "Validação e diagnóstico dos campos da IR.",
    },
    {
        "funcao": "assert_valid_ir",
        "entrada": "IR + JSON Schema",
        "saida": "nenhuma",
        "finalidade": "Interrupção imediata quando a IR é inválida.",
    },
    {
        "funcao": "canonical_ir_json",
        "entrada": "IR",
        "saida": "JSON canônico",
        "finalidade": "Serialização estável para hash e auditoria.",
    },
    {
        "funcao": "summarize_ir",
        "entrada": "IR",
        "saida": "resumo",
        "finalidade": "Extração de campos de apresentação e controle.",
    },
])

exibir_tabela(
    funcoes_modulo,
    "Funções públicas do módulo ir_core.py",
    altura_px=420,
)


# ----------------------------------------------------------
# 5.6 Saída do bloco
# ----------------------------------------------------------

print("Bloco 5 concluído")
print(f"Módulo salvo em: {IR_CORE_PATH}")
print(f"Teste rápido executado com: {id_teste_rapido}")
print("Status: OK")

função,entrada,saída,finalidade
ast_to_ir,AST,IR,Transformação determinística da estrutura do parser.
ir_to_ast,IR,AST,Reconstrução determinística da estrutura sintática abstrata.
validate_ir,IR + JSON Schema,resultado estruturado,Validação e diagnóstico dos campos da IR.
assert_valid_ir,IR + JSON Schema,nenhuma,Interrupção imediata quando a IR é inválida.
canonical_ir_json,IR,JSON canônico,Serialização estável para hash e auditoria.
summarize_ir,IR,resumo,Extração de campos de apresentação e controle.


Bloco 5 concluído
Módulo salvo em: /kaggle/working/f1_operacional/ir_core.py
Teste rápido executado com: campi_001
Status: OK


## Bloco 6 - Geração determinística das 50 IRs

### Objetivo

Este bloco aplica a transformação `ast_to_ir` a todas as referências canônicas do CAMPI e persiste uma IR por exemplo.

### Processo de geração

Para cada ID:

1. a Nile canônica é analisada pelo parser da F0;
2. a AST é recuperada;
3. `ast_to_ir` produz a estrutura intermediária;
4. a IR é validada imediatamente pelo JSON Schema;
5. o objeto é serializado de forma canônica;
6. o registro é associado ao ID original.

A execução é interrompida no primeiro erro, evitando a produção de um arquivo parcialmente válido.

### Artefato produzido

As 50 IRs são armazenadas em:

```text
ir_references.jsonl
```

Cada linha contém um registro lógico independente com:

- `id`;
- `ir`.

A entrada em linguagem natural e a Nile não são duplicadas nesse arquivo, pois permanecem na F0 e podem ser relacionadas pelo ID.

### Verificações

O bloco confirma:

- exatamente 50 registros;
- IDs de `campi_001` a `campi_050`;
- ausência de duplicações;
- conformidade de todas as IRs;
- ordem estável;
- releitura integral do JSONL persistido.

### Interpretação

As IRs não são respostas de modelo e não passaram por edição manual. Elas são derivações determinísticas das referências Nile.

### Resultado esperado

A tabela apresenta uma visão resumida das 50 estruturas. O arquivo persistido passa a ser a fonte oficial das IRs de referência.

In [6]:
# ----------------------------------------------------------
# 6.1 Geração das IRs a partir das ASTs
# ----------------------------------------------------------

registros_ir = []
resumos_ir = []

for linha in df_campi.itertuples(index=False):
    ast_referencia = asts_por_id[linha.id]
    ir = ir_core.ast_to_ir(ast_referencia)
    validacao = ir_core.validate_ir(ir, ir_schema_runtime)

    if not validacao["schema_valid"]:
        raise RuntimeError(
            f"A IR de {linha.id} foi rejeitada pelo esquema: "
            + json.dumps(validacao["schema_errors"], ensure_ascii=False)
        )

    json_canonico_ir = ir_core.canonical_ir_json(ir)

    registros_ir.append({
        "id": linha.id,
        "ir": ir,
    })

    resumo = ir_core.summarize_ir(ir)
    resumo.update({
        "id": linha.id,
        "ir_sha256": calcular_sha256_texto(json_canonico_ir),
    })
    resumos_ir.append(resumo)


# ----------------------------------------------------------
# 6.2 Verificação da quantidade e dos identificadores
# ----------------------------------------------------------

if len(registros_ir) != EXPECTED_EXAMPLES:
    raise ValueError(
        f"Foram produzidas {len(registros_ir)} IRs. "
        f"Esperado: {EXPECTED_EXAMPLES}."
    )

ids_ir = [registro["id"] for registro in registros_ir]

if len(ids_ir) != len(set(ids_ir)):
    raise ValueError("Foram produzidos identificadores de IR duplicados.")

if ids_ir != df_campi["id"].tolist():
    raise ValueError("A ordem ou o conjunto de IDs das IRs difere do CAMPI canônico.")


# ----------------------------------------------------------
# 6.3 Salvamento e releitura do arquivo JSONL
# ----------------------------------------------------------

salvar_jsonl(registros_ir, IR_REFERENCES_PATH)
registros_ir_recarregados = ler_jsonl(IR_REFERENCES_PATH)

if registros_ir_recarregados != registros_ir:
    raise RuntimeError("A releitura de ir_references.jsonl não preservou os registros.")


# ----------------------------------------------------------
# 6.4 Tabela resumida das 50 IRs
# ----------------------------------------------------------

df_resumo_ir = pd.DataFrame(resumos_ir)[[
    "id",
    "intent_id",
    "scope_type",
    "source",
    "destination",
    "targets",
    "n_targets",
    "n_operations",
    "operators",
    "n_items",
    "has_temporal",
    "ir_sha256",
]]

exibir_tabela(
    df_resumo_ir,
    "Resumo das 50 Representações Intermediárias geradas",
    altura_px=560,
)


# ----------------------------------------------------------
# 6.5 Saída do bloco
# ----------------------------------------------------------

print("Bloco 6 concluído")
print(f"IRs geradas: {len(registros_ir)}")
print(f"Arquivo salvo em: {IR_REFERENCES_PATH}")
print("Status: OK")

ID,intent ID,scope type,source,destination,targets,quantidade de targets,quantidade de operations,operators,quantidade de itens,possui temporal constraint,SHA-256 da IR
campi_001,uniIntent,for,-,-,group('students'),1,1,add,1,não,3b1e06fd737fe1a7967ae00e202952286a4014a125637386dabf7ad3f3ab799a
campi_002,uniIntent,for,-,-,endpoint('university'),1,1,unset,1,não,ef372a8b1affd0d6279171956c3d13d230fc18ea85ac5d246cdf3136d9c10d36
campi_003,uniIntent,for,-,-,endpoint('dorms'),1,1,add,1,não,c6a7dd1ec0c6d083d8b6e642be74e802862f934568a3670a5e4dbb7305d9c80c
campi_004,uniIntent,for,-,-,endpoint('university'),1,2,add | block,2,não,03c3a9538e035b127f7aada7e4e035ebf24e7ca140c6513727acd636de0b5fde
campi_005,uniIntent,for,-,-,endpoint('university'),1,2,add | allow,3,não,7cdbe9dec58c1c7166037205846c3eb25d59b86bcb86ee1d8ac8b8b5a597348f
campi_006,uniIntent,for,-,-,endpoint('university'),1,2,add | allow,2,não,91e59bbc15334cc1ee7bee7c568a639a8192b4a46630a215cdf58d5b5cffa85e
campi_007,uniIntent,for,-,-,endpoint('university'),1,2,add | allow,2,não,b957f22f24c1d283a2e0807ae9b44159265903f6a3e0e0507af6ca49d77cbc8a
campi_008,uniIntent,for,-,-,endpoint('university'),1,2,add | block,2,não,b6439827693d8bc130c9a95d539c6e94284660c2f845e033bd08a42d246620c2
campi_009,uniIntent,for,-,-,endpoint('university'),1,2,add | allow,3,não,7725dff71c21b78c1b7bedb5cba051b8a1f9bafaa61eb1edc1e1d3ef57603540
campi_010,uniIntent,for,-,-,endpoint('university'),1,2,add | allow,3,não,c2e898c57449bdfdf61dfcaa71367a0d82056a274b09fdc70db55ad07f3e74b2


Bloco 6 concluído
IRs geradas: 50
Arquivo salvo em: /kaggle/working/f1_operacional/ir_references.jsonl
Status: OK


## Bloco 7 - Validação integral das IRs

### Objetivo

Este bloco valida as IRs como artefatos persistidos. A releitura de `ir_references.jsonl` impede que a validação dependa apenas dos objetos mantidos em memória durante a geração.

### Conferências por registro

Para cada linha, são verificados:

- presença de `id` e `ir`;
- unicidade do ID;
- correspondência com os IDs da F0;
- conformidade com `ir_schema.json`;
- quantidade de targets e operações;
- tipos e argumentos dos itens;
- presença ou ausência coerente de `temporal_constraint`;
- hash da serialização canônica.

### Cobertura global

O bloco confirma:

- 50 IRs;
- nenhum ID ausente;
- nenhum ID adicional;
- nenhuma duplicação;
- nenhuma falha de schema.

### Casos negativos dirigidos

São construídas IRs inválidas para testar rejeições como:

- campo obrigatório ausente;
- propriedade adicional;
- valor fora do vocabulário;
- combinação incompatível de escopo;
- operador ou item inválido;
- argumento ausente;
- horário fora do formato;
- restrição temporal incompleta.

Esses casos testam classes específicas do schema e funcionam como regressão. Não constituem prova de completude para todo JSON possível.

### Tabelas produzidas

O bloco exibe a validação das 50 referências e o resultado dos casos negativos.

### Resultado esperado

Todas as IRs de referência devem ser aceitas e todos os casos negativos devem ser rejeitados conforme a expectativa.

In [7]:
# ----------------------------------------------------------
# 7.1 Validação dos registros persistidos
# ----------------------------------------------------------

registros_persistidos = ler_jsonl(IR_REFERENCES_PATH)
linhas_validacao_ir = []

for registro in registros_persistidos:
    if not isinstance(registro, dict) or set(registro) != {"id", "ir"}:
        raise ValueError(
            "Cada linha de ir_references.jsonl deve conter somente os campos id e ir."
        )

    id_exemplo = registro["id"]
    ir = registro["ir"]
    resultado = ir_core.validate_ir(ir, ir_schema_runtime)
    resumo = ir_core.summarize_ir(ir)

    linhas_validacao_ir.append({
        "id": id_exemplo,
        "schema_valid": resultado["schema_valid"],
        "schema_error_count": len(resultado["schema_errors"]),
        "schema_feedback": (
            "IR válida."
            if resultado["schema_valid"]
            else " | ".join(
                f"{erro['path']}: {erro['message']}"
                for erro in resultado["schema_errors"]
            )
        ),
        "scope_type": resumo["scope_type"],
        "source": resumo["source"],
        "destination": resumo["destination"],
        "n_targets": resumo["n_targets"],
        "n_operations": resumo["n_operations"],
        "operators": resumo["operators"],
        "n_items": resumo["n_items"],
        "has_temporal": resumo["has_temporal"],
        "ir_sha256": calcular_sha256_texto(ir_core.canonical_ir_json(ir)),
    })

df_validacao_ir = pd.DataFrame(linhas_validacao_ir)

if len(df_validacao_ir) != EXPECTED_EXAMPLES:
    raise ValueError("A validação não processou as 50 IRs.")

if df_validacao_ir["id"].nunique() != EXPECTED_EXAMPLES:
    raise ValueError("Os IDs do arquivo de IRs não são únicos.")

if set(df_validacao_ir["id"]) != set(df_campi["id"]):
    raise ValueError("Os IDs das IRs não coincidem com os IDs do CAMPI.")

if not df_validacao_ir["schema_valid"].all():
    invalidas = df_validacao_ir.loc[~df_validacao_ir["schema_valid"]]
    raise RuntimeError(
        "Foram encontradas IRs inválidas:\n"
        + invalidas.to_string(index=False)
    )


# ----------------------------------------------------------
# 7.2 Construção de casos negativos do esquema
# ----------------------------------------------------------

ir_base_negativa = copy.deepcopy(registros_persistidos[0]["ir"])

casos_negativos_ir = []

caso = copy.deepcopy(ir_base_negativa)
del caso["intent_id"]
casos_negativos_ir.append(("campo_obrigatorio_ausente", caso))

caso = copy.deepcopy(ir_base_negativa)
caso["campo_extra"] = "valor"
casos_negativos_ir.append(("propriedade_adicional", caso))

caso = copy.deepcopy(ir_base_negativa)
caso["scope"]["type"] = "unknown"
casos_negativos_ir.append(("scope_type_invalido", caso))

caso = copy.deepcopy(ir_base_negativa)
caso["scope"]["source"] = {"kind": "endpoint", "value": "campus"}
casos_negativos_ir.append(("source_incompativel_com_for", caso))

caso = copy.deepcopy(ir_base_negativa)
caso["scope"]["targets"] = []
casos_negativos_ir.append(("for_sem_target", caso))

caso = copy.deepcopy(ir_base_negativa)
caso["operations"][0]["operator"] = "remove"
casos_negativos_ir.append(("operator_fora_vocabulario", caso))

caso = copy.deepcopy(ir_base_negativa)
caso["operations"] = [{
    "operator": "set",
    "items": [{
        "kind": "bandwidth",
        "constraint": "max",
        "value": "10",
        "unit": "gb/d",
    }],
}]
casos_negativos_ir.append(("unidade_incompativel", caso))

caso = copy.deepcopy(ir_base_negativa)
caso["operations"] = [{
    "operator": "unset",
    "items": [{"kind": "quota"}],
}]
casos_negativos_ir.append(("unset_kind_invalido", caso))

caso = copy.deepcopy(ir_base_negativa)
caso["temporal_constraint"] = {"start": "25:00", "end": "26:00"}
casos_negativos_ir.append(("horario_invalido", caso))

caso = copy.deepcopy(ir_base_negativa)
caso["operations"][0]["items"][0]["value"] = ""
casos_negativos_ir.append(("valor_vazio", caso))

caso = copy.deepcopy(ir_base_negativa)
caso["operations"][0]["items"][0]["value"] = "valor\nquebrado"
casos_negativos_ir.append(("quebra_de_linha_em_valor", caso))

caso = copy.deepcopy(ir_base_negativa)
caso["operations"][0]["items"][0]["value"] = "student's network"
casos_negativos_ir.append(("apostrofo_em_valor", caso))


# ----------------------------------------------------------
# 7.3 Execução dos casos negativos
# ----------------------------------------------------------

resultados_negativos_ir = []

for nome_caso, ir_invalida in casos_negativos_ir:
    resultado = ir_core.validate_ir(ir_invalida, ir_schema_runtime)
    rejeitada = not resultado["schema_valid"]

    resultados_negativos_ir.append({
        "categoria": nome_caso,
        "resultado_esperado": "rejeitada",
        "resultado_observado": "rejeitada" if rejeitada else "aceita",
        "aprovados": rejeitada,
    })

df_testes_negativos_ir = pd.DataFrame(resultados_negativos_ir)

if not df_testes_negativos_ir["aprovados"].all():
    falhas = df_testes_negativos_ir.loc[~df_testes_negativos_ir["aprovados"]]
    raise AssertionError(
        "O esquema aceitou caso negativo que deveria ser rejeitado:\n"
        + falhas.to_string(index=False)
    )


# ----------------------------------------------------------
# 7.4 Tabelas de validação
# ----------------------------------------------------------

resumo_validacao_schema = pd.DataFrame([
    {
        "categoria": "IRs persistidas",
        "casos": len(df_validacao_ir),
        "aprovados": int(df_validacao_ir["schema_valid"].sum()),
        "status": "OK",
    },
    {
        "categoria": "casos negativos dirigidos",
        "casos": len(df_testes_negativos_ir),
        "aprovados": int(df_testes_negativos_ir["aprovados"].sum()),
        "status": "OK",
    },
])

exibir_tabela(
    resumo_validacao_schema,
    "Resumo da validação pelo JSON Schema",
    altura_px=260,
)

exibir_tabela(
    df_validacao_ir,
    "Validação individual das 50 Representações Intermediárias",
    altura_px=560,
)

exibir_tabela(
    df_testes_negativos_ir,
    "Casos negativos dirigidos do esquema da IR",
    altura_px=430,
)


# ----------------------------------------------------------
# 7.5 Saída do bloco
# ----------------------------------------------------------

print("Bloco 7 concluído")
print(f"IRs aprovadas pelo esquema: {int(df_validacao_ir['schema_valid'].sum())}/50")
print(f"Casos negativos rejeitados: {int(df_testes_negativos_ir['aprovados'].sum())}/{len(df_testes_negativos_ir)}")
print("Status: OK")

categoria,casos,aprovados,status
IRs persistidas,50,50,OK
casos negativos dirigidos,12,12,OK


ID,esquema válido,erros de esquema,diagnóstico do esquema,scope type,source,destination,quantidade de targets,quantidade de operations,operators,quantidade de itens,possui temporal constraint,SHA-256 da IR
campi_001,sim,0,IR válida.,for,-,-,1,1,add,1,não,3b1e06fd737fe1a7967ae00e202952286a4014a125637386dabf7ad3f3ab799a
campi_002,sim,0,IR válida.,for,-,-,1,1,unset,1,não,ef372a8b1affd0d6279171956c3d13d230fc18ea85ac5d246cdf3136d9c10d36
campi_003,sim,0,IR válida.,for,-,-,1,1,add,1,não,c6a7dd1ec0c6d083d8b6e642be74e802862f934568a3670a5e4dbb7305d9c80c
campi_004,sim,0,IR válida.,for,-,-,1,2,add | block,2,não,03c3a9538e035b127f7aada7e4e035ebf24e7ca140c6513727acd636de0b5fde
campi_005,sim,0,IR válida.,for,-,-,1,2,add | allow,3,não,7cdbe9dec58c1c7166037205846c3eb25d59b86bcb86ee1d8ac8b8b5a597348f
campi_006,sim,0,IR válida.,for,-,-,1,2,add | allow,2,não,91e59bbc15334cc1ee7bee7c568a639a8192b4a46630a215cdf58d5b5cffa85e
campi_007,sim,0,IR válida.,for,-,-,1,2,add | allow,2,não,b957f22f24c1d283a2e0807ae9b44159265903f6a3e0e0507af6ca49d77cbc8a
campi_008,sim,0,IR válida.,for,-,-,1,2,add | block,2,não,b6439827693d8bc130c9a95d539c6e94284660c2f845e033bd08a42d246620c2
campi_009,sim,0,IR válida.,for,-,-,1,2,add | allow,3,não,7725dff71c21b78c1b7bedb5cba051b8a1f9bafaa61eb1edc1e1d3ef57603540
campi_010,sim,0,IR válida.,for,-,-,1,2,add | allow,3,não,c2e898c57449bdfdf61dfcaa71367a0d82056a274b09fdc70db55ad07f3e74b2


categoria,resultado esperado,resultado observado,aprovados
campo_obrigatorio_ausente,rejeitada,rejeitada,sim
propriedade_adicional,rejeitada,rejeitada,sim
scope_type_invalido,rejeitada,rejeitada,sim
source_incompativel_com_for,rejeitada,rejeitada,sim
for_sem_target,rejeitada,rejeitada,sim
operator_fora_vocabulario,rejeitada,rejeitada,sim
unidade_incompativel,rejeitada,rejeitada,sim
unset_kind_invalido,rejeitada,rejeitada,sim
horario_invalido,rejeitada,rejeitada,sim
valor_vazio,rejeitada,rejeitada,sim


Bloco 7 concluído
IRs aprovadas pelo esquema: 50/50
Casos negativos rejeitados: 12/12
Status: OK


## Bloco 8 - Verificação de equivalência entre AST e IR

### Objetivo

Este bloco verifica se a IR preserva integralmente a estrutura necessária para reconstruir as referências do CAMPI.

### Cadeia de equivalência

Para cada exemplo, são executadas as etapas:

```text
AST original
    ↓ ast_to_ir
IR persistida
    ↓ ir_to_ast
AST reconstruída
    ↓ renderer
Nile reconstruída
    ↓ parser
AST da segunda análise
```

São realizadas três comparações:

1. AST original versus AST reconstruída;
2. Nile reconstruída versus Nile canônica;
3. AST original versus AST obtida após a segunda análise.

### Significado da verificação

Quando as três comparações são verdadeiras, a IR preserva, para aquele exemplo, todos os componentes formais necessários à ida e volta. A verificação é executada sobre as 50 referências concretas e não implica prova geral para estruturas fora do subconjunto estudado.

### Consolidação

Os resultados são registrados em:

```text
ir_validation.csv
```

O arquivo inclui estado do schema, equivalência da AST, correspondência da Nile e hashes relevantes.

### Barreiras

A execução falha quando:

- uma IR não pode ser reconstruída;
- a AST reconstruída diverge;
- a Nile renderizada diverge da referência;
- a nova análise produz estrutura diferente;
- a quantidade de registros não é 50.

### Resultado esperado

Todas as 50 cadeias devem ser equivalentes. As tabelas finais apresentam os resultados por exemplo e o resumo global.

In [8]:
# ----------------------------------------------------------
# 8.1 Indexação das IRs persistidas
# ----------------------------------------------------------

ir_por_id = {
    registro["id"]: registro["ir"]
    for registro in registros_persistidos
}

if len(ir_por_id) != EXPECTED_EXAMPLES:
    raise ValueError("A indexação das IRs não contém os 50 exemplos esperados.")


# ----------------------------------------------------------
# 8.2 Verificação AST → IR → AST → Nile → AST
# ----------------------------------------------------------

linhas_equivalencia = []

for linha in df_campi.itertuples(index=False):
    ast_original = asts_por_id[linha.id]
    ir = ir_por_id[linha.id]
    ast_reconstruida = ir_core.ir_to_ast(ir)

    ast_equivalent = ast_reconstruida == ast_original
    nile_reconstruida = nile_core.render_ast(ast_reconstruida)
    canonical_nile_equal = nile_reconstruida == linha.nile_canonical

    resultado_nile_reconstruida = validator_nile.validate(nile_reconstruida)
    nile_roundtrip_ok = (
        resultado_nile_reconstruida["valid"]
        and resultado_nile_reconstruida["ast"] == ast_original
    )

    linhas_equivalencia.append({
        "id": linha.id,
        "ast_equivalent": ast_equivalent,
        "canonical_nile_equal": canonical_nile_equal,
        "nile_roundtrip_ok": nile_roundtrip_ok,
    })


df_equivalencia = pd.DataFrame(linhas_equivalencia)

colunas_equivalencia = [
    "ast_equivalent",
    "canonical_nile_equal",
    "nile_roundtrip_ok",
]

if not df_equivalencia[colunas_equivalencia].all().all():
    falhas = df_equivalencia.loc[
        ~df_equivalencia[colunas_equivalencia].all(axis=1)
    ]
    raise AssertionError(
        "A equivalência AST-IR falhou para uma ou mais referências:\n"
        + falhas.to_string(index=False)
    )


# ----------------------------------------------------------
# 8.3 Consolidação do arquivo final de validação
# ----------------------------------------------------------

df_validacao_final = df_validacao_ir.merge(
    df_equivalencia,
    on="id",
    how="left",
    validate="one_to_one",
)

if df_validacao_final[colunas_equivalencia].isna().any().any():
    raise RuntimeError("A consolidação deixou resultados de equivalência ausentes.")

df_validacao_final.to_csv(
    IR_VALIDATION_PATH,
    index=False,
    encoding="utf-8",
)

validacao_recarregada = pd.read_csv(IR_VALIDATION_PATH)

if len(validacao_recarregada) != EXPECTED_EXAMPLES:
    raise RuntimeError("O arquivo ir_validation.csv não contém 50 registros.")


# ----------------------------------------------------------
# 8.4 Tabelas de equivalência
# ----------------------------------------------------------

resumo_equivalencia = pd.DataFrame([
    {
        "categoria": "AST original = AST reconstruída",
        "casos": len(df_equivalencia),
        "aprovados": int(df_equivalencia["ast_equivalent"].sum()),
        "status": "OK",
    },
    {
        "categoria": "Nile canônica preservada",
        "casos": len(df_equivalencia),
        "aprovados": int(df_equivalencia["canonical_nile_equal"].sum()),
        "status": "OK",
    },
    {
        "categoria": "Nile reconstruída aceita e gera a AST original",
        "casos": len(df_equivalencia),
        "aprovados": int(df_equivalencia["nile_roundtrip_ok"].sum()),
        "status": "OK",
    },
])

exibir_tabela(
    resumo_equivalencia,
    "Resumo da equivalência entre AST, IR e Nile",
    altura_px=300,
)

exibir_tabela(
    df_validacao_final,
    "Validação final e equivalência das 50 Representações Intermediárias",
    altura_px=570,
)


# ----------------------------------------------------------
# 8.5 Saída do bloco
# ----------------------------------------------------------

print("Bloco 8 concluído")
print(f"ASTs equivalentes: {int(df_equivalencia['ast_equivalent'].sum())}/50")
print(f"Nile canônica preservada: {int(df_equivalencia['canonical_nile_equal'].sum())}/50")
print(f"Idas e voltas Nile aprovadas: {int(df_equivalencia['nile_roundtrip_ok'].sum())}/50")
print(f"Arquivo salvo em: {IR_VALIDATION_PATH}")
print("Status: OK")

categoria,casos,aprovados,status
AST original = AST reconstruída,50,50,OK
Nile canônica preservada,50,50,OK
Nile reconstruída aceita e gera a AST original,50,50,OK


ID,esquema válido,erros de esquema,diagnóstico do esquema,scope type,source,destination,quantidade de targets,quantidade de operations,operators,quantidade de itens,possui temporal constraint,SHA-256 da IR,AST equivalente,Nile canônica preservada,ida e volta Nile válida
campi_001,sim,0,IR válida.,for,-,-,1,1,add,1,não,3b1e06fd737fe1a7967ae00e202952286a4014a125637386dabf7ad3f3ab799a,sim,sim,sim
campi_002,sim,0,IR válida.,for,-,-,1,1,unset,1,não,ef372a8b1affd0d6279171956c3d13d230fc18ea85ac5d246cdf3136d9c10d36,sim,sim,sim
campi_003,sim,0,IR válida.,for,-,-,1,1,add,1,não,c6a7dd1ec0c6d083d8b6e642be74e802862f934568a3670a5e4dbb7305d9c80c,sim,sim,sim
campi_004,sim,0,IR válida.,for,-,-,1,2,add | block,2,não,03c3a9538e035b127f7aada7e4e035ebf24e7ca140c6513727acd636de0b5fde,sim,sim,sim
campi_005,sim,0,IR válida.,for,-,-,1,2,add | allow,3,não,7cdbe9dec58c1c7166037205846c3eb25d59b86bcb86ee1d8ac8b8b5a597348f,sim,sim,sim
campi_006,sim,0,IR válida.,for,-,-,1,2,add | allow,2,não,91e59bbc15334cc1ee7bee7c568a639a8192b4a46630a215cdf58d5b5cffa85e,sim,sim,sim
campi_007,sim,0,IR válida.,for,-,-,1,2,add | allow,2,não,b957f22f24c1d283a2e0807ae9b44159265903f6a3e0e0507af6ca49d77cbc8a,sim,sim,sim
campi_008,sim,0,IR válida.,for,-,-,1,2,add | block,2,não,b6439827693d8bc130c9a95d539c6e94284660c2f845e033bd08a42d246620c2,sim,sim,sim
campi_009,sim,0,IR válida.,for,-,-,1,2,add | allow,3,não,7725dff71c21b78c1b7bedb5cba051b8a1f9bafaa61eb1edc1e1d3ef57603540,sim,sim,sim
campi_010,sim,0,IR válida.,for,-,-,1,2,add | allow,3,não,c2e898c57449bdfdf61dfcaa71367a0d82056a274b09fdc70db55ad07f3e74b2,sim,sim,sim


Bloco 8 concluído
ASTs equivalentes: 50/50
Nile canônica preservada: 50/50
Idas e voltas Nile aprovadas: 50/50
Arquivo salvo em: /kaggle/working/f1_operacional/ir_validation.csv
Status: OK


## Bloco 9 - Manifesto e empacotamento final

### Objetivo

Este bloco encerra a F1 e cria o pacote operacional consumido pela F2 e pela F3.

### Limpeza e auditoria

Antes do manifesto, o bloco:

- remove caches e bytecodes;
- confirma os cinco artefatos obrigatórios;
- verifica que as 50 IRs estão presentes;
- confirma os resultados da validação e da equivalência;
- relê os arquivos persistidos.

### Manifesto

`manifest.json` registra:

- fase, conjunto e versão da IR;
- hash do manifesto da F0;
- hashes dos artefatos da F0 efetivamente utilizados;
- método determinístico de construção;
- quantidade de IRs;
- resultado do JSON Schema;
- equivalência AST ↔ IR;
- ida e volta Nile;
- versões das bibliotecas;
- tamanho e SHA-256 dos arquivos produzidos.

### Pacote final

O ZIP contém exatamente:

```text
f1_operacional.zip
├── ir_schema.json
├── ir_core.py
├── ir_references.jsonl
├── ir_validation.csv
└── manifest.json
```

Arquivos temporários não são incluídos.

### Verificação posterior

Depois da criação, o ZIP é reaberto. O bloco compara sua lista de arquivos com a estrutura esperada e confere os hashes registrados no manifesto.

### Limites

O pacote documenta equivalência operacional para as 50 referências do CAMPI. Ele não declara que a IR cobre toda a linguagem Nile.

### Resultado esperado

As tabelas finais apresentam o inventário e o resumo da fase. A F1 só é considerada concluída quando o ZIP possui exatamente os cinco arquivos previstos e todos os hashes conferem.

In [9]:
# ----------------------------------------------------------
# 9.1 Remoção de arquivos temporários
# ----------------------------------------------------------

for diretorio_pycache in F1_DIR.rglob("__pycache__"):
    shutil.rmtree(diretorio_pycache)

for arquivo_pyc in F1_DIR.rglob("*.pyc"):
    arquivo_pyc.unlink()


# ----------------------------------------------------------
# 9.2 Verificação dos artefatos obrigatórios antes do manifesto
# ----------------------------------------------------------

artefatos_f1_sem_manifesto = [
    IR_SCHEMA_PATH,
    IR_CORE_PATH,
    IR_REFERENCES_PATH,
    IR_VALIDATION_PATH,
]

faltantes_f1 = [
    str(caminho)
    for caminho in artefatos_f1_sem_manifesto
    if not caminho.is_file()
]

if faltantes_f1:
    raise FileNotFoundError(
        "Artefatos obrigatórios da F1 ausentes:\n"
        + "\n".join(faltantes_f1)
    )


# ----------------------------------------------------------
# 9.3 Construção do manifesto da F1
# ----------------------------------------------------------

arquivos_manifesto = []

for caminho in sorted(artefatos_f1_sem_manifesto, key=lambda p: p.name):
    arquivos_manifesto.append({
        "arquivo": caminho.relative_to(F1_DIR).as_posix(),
        "tamanho_bytes": caminho.stat().st_size,
        "sha256": calcular_sha256(caminho),
    })

manifesto_f1 = {
    "fase": FASE,
    "descricao": "Construção determinística da Representação Intermediária do CAMPI.",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "input": {
        "fase": "F0",
        "dataset": DATASET_ID,
        "manifest_file": "manifest.json",
        "manifest_sha256": calcular_sha256(F0_MANIFEST_PATH),
        "campi_canonical_file": F0_CAMPI_PATH.name,
        "campi_canonical_sha256": calcular_sha256(F0_CAMPI_PATH),
        "grammar_file": F0_GRAMMAR_PATH.name,
        "grammar_sha256": calcular_sha256(F0_GRAMMAR_PATH),
        "nile_core_file": F0_NILE_CORE_PATH.name,
        "nile_core_sha256": calcular_sha256(F0_NILE_CORE_PATH),
        "reference_validation_file": F0_REFERENCE_VALIDATION_PATH.name,
        "reference_validation_sha256": calcular_sha256(F0_REFERENCE_VALIDATION_PATH),
    },
    "ir": {
        "version": IR_VERSION,
        "construction": "deterministic_ast_to_ir",
        "inverse_transformation": "deterministic_ir_to_ast",
        "schema_validation": "Draft 2020-12",
        "schema_file": IR_SCHEMA_PATH.name,
        "core_module": IR_CORE_PATH.name,
        "references_file": IR_REFERENCES_PATH.name,
        "validation_file": IR_VALIDATION_PATH.name,
        "total_examples": len(registros_persistidos),
        "schema_valid_examples": int(df_validacao_final["schema_valid"].sum()),
        "ast_equivalent_examples": int(df_validacao_final["ast_equivalent"].sum()),
        "canonical_nile_preserved_examples": int(df_validacao_final["canonical_nile_equal"].sum()),
        "nile_roundtrip_examples": int(df_validacao_final["nile_roundtrip_ok"].sum()),
        "negative_schema_tests": len(df_testes_negativos_ir),
        "negative_schema_tests_passed": int(df_testes_negativos_ir["aprovados"].sum()),
    },
    "method": {
        "teacher_model_used": False,
        "student_model_used": False,
        "manual_ir_generation": False,
        "ast_persisted": False,
        "ir_persisted": True,
    },
    "coverage": {
        "scope_types": dict(sorted(contagem_scope.items())),
        "operators": dict(sorted(contagem_operator.items())),
        "item_kinds": dict(sorted(contagem_kind.items())),
        "temporal_constraints": dict(sorted(contagem_temporal.items())),
    },
    "environment": {
        "python": platform.python_version(),
        "platform": platform.platform(),
        "pandas": versao_pacote("pandas"),
        "jsonschema": versao_pacote("jsonschema"),
    },
    "files": arquivos_manifesto,
}

salvar_json(manifesto_f1, MANIFEST_PATH)


# ----------------------------------------------------------
# 9.4 Criação do pacote ZIP
# ----------------------------------------------------------

arquivos_para_zip = sorted(
    [caminho for caminho in F1_DIR.rglob("*") if caminho.is_file()],
    key=lambda p: p.relative_to(F1_DIR).as_posix(),
)

if len(arquivos_para_zip) != EXPECTED_OUTPUT_FILES:
    raise RuntimeError(
        f"Quantidade inesperada de arquivos na F1: {len(arquivos_para_zip)}. "
        f"Esperado: {EXPECTED_OUTPUT_FILES}."
    )

with zipfile.ZipFile(ZIP_F1_PATH, "w", compression=zipfile.ZIP_DEFLATED) as arquivo_zip:
    for caminho in arquivos_para_zip:
        arquivo_zip.write(
            caminho,
            arcname=caminho.relative_to(F1_DIR).as_posix(),
        )


# ----------------------------------------------------------
# 9.5 Verificação do conteúdo do ZIP
# ----------------------------------------------------------

with zipfile.ZipFile(ZIP_F1_PATH, "r") as arquivo_zip:
    nomes_zip = sorted(arquivo_zip.namelist())

nomes_esperados = sorted(
    caminho.relative_to(F1_DIR).as_posix()
    for caminho in arquivos_para_zip
)

if nomes_zip != nomes_esperados:
    raise RuntimeError("O conteúdo do ZIP difere dos artefatos da F1.")

if any("__pycache__" in nome or nome.endswith(".pyc") for nome in nomes_zip):
    raise RuntimeError("O ZIP contém bytecode ou diretório temporário.")


# ----------------------------------------------------------
# 9.6 Conferência dos hashes do manifesto
# ----------------------------------------------------------

manifesto_recarregado = ler_json(MANIFEST_PATH)

for item in manifesto_recarregado["files"]:
    caminho = F1_DIR / item["arquivo"]
    if calcular_sha256(caminho) != item["sha256"]:
        raise RuntimeError(
            f"Hash divergente após a gravação do manifesto: {item['arquivo']}"
        )


# ----------------------------------------------------------
# 9.7 Tabelas finais
# ----------------------------------------------------------

arquivos_finais = pd.DataFrame([
    {
        "arquivo": caminho.relative_to(F1_DIR).as_posix(),
        "tamanho_bytes": caminho.stat().st_size,
        "sha256": calcular_sha256(caminho),
    }
    for caminho in arquivos_para_zip
])

resumo_final = pd.DataFrame([{
    "fase": FASE,
    "dataset": DATASET_ID,
    "exemplos": len(registros_persistidos),
    "irs_validas": int(df_validacao_final["schema_valid"].sum()),
    "equivalencias_ast_ir": int(df_validacao_final["ast_equivalent"].sum()),
    "idas_e_voltas_nile": int(df_validacao_final["nile_roundtrip_ok"].sum()),
    "arquivos_no_zip": len(nomes_zip),
    "zip": ZIP_F1_PATH.name,
    "status": "OK",
}])

exibir_tabela(
    arquivos_finais,
    "Artefatos finais da F1",
    altura_px=360,
)

exibir_tabela(
    resumo_final,
    "Resumo final da F1",
    altura_px=240,
)


# ----------------------------------------------------------
# 9.8 Saída do bloco
# ----------------------------------------------------------

print("Bloco 9 concluído")
print(f"ZIP gerado: {ZIP_F1_PATH}")
print(f"Tamanho do ZIP: {ZIP_F1_PATH.stat().st_size:,} bytes")
print("F1 concluída com sucesso")
print("Status: OK")

arquivo,tamanho em bytes,sha256
ir_core.py,4812,45f4d8602cb3fc3dcfd119f24e7eecac137eee385277e197ab42a2662744685b
ir_references.jsonl,17967,1572888aadd6cb05798c6ba3ea1c8a84c9c6110199275d77d45566a93330253c
ir_schema.json,9174,9d34bc405a490626b09833f041dc7229ddcd40cfa12608dca6f4c58627a48d98
ir_validation.csv,6952,e26b6997109d459e7fe0055f5ed23c8e41370c8eaa99e3a124aeed282738cc6d
manifest.json,2956,85f32e9d14b7e24a8fba5c411625a6ea8e43192526db8109f68a6bacd68d6e5f


fase,conjunto de dados,exemplos,IRs válidas,equivalências AST-IR,idas e voltas Nile,arquivos no ZIP,ZIP,status
F1,CAMPI,50,50,50,50,5,f1_operacional.zip,OK


Bloco 9 concluído
ZIP gerado: /kaggle/working/f1_operacional.zip
Tamanho do ZIP: 8,103 bytes
F1 concluída com sucesso
Status: OK
